# TAES-2026-0295 R8: common-recovery reproduction
Upload the GitHub code/data ZIP manually. This notebook reproduces the 504-run resource-sensitivity
campaign from bundled derived calibration inputs and verifies numerical outputs.
The original R6 failures remain in the numerical records. No flight-hardware or formal
stability validation is implied. Estimated duration depends on Colab resources.


In [ ]:
import sys, subprocess
packages = ["numpy==2.3.5", "scipy==1.17.0", "pandas==2.2.3", "matplotlib==3.10.8",
            "pymsis==0.13.0", "quadprog==0.1.13", "spaceweather==0.4.2"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archive = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if archive is None:
    raise ValueError("Upload the R8 GitHub code/data ZIP")
work = Path("/content/taes_r8"); work.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    if z.testzip() is not None:
        raise ValueError("ZIP integrity check failed")
    z.extractall(work)
runner = next(work.rglob("common_recovery_campaign.py"))
root = runner.parent.parent
os.chdir(runner.parent)
print("Working in", root)


In [ ]:
env = os.environ.copy()
env.update(OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", NPROC="2")
for script in ("resource_sensitivity.py", "verify_resource_sensitivity.py",
               "make_figures.py", "verify_common_recovery.py"):
    args = [sys.executable, script]
    if script == "make_figures.py":
        args += ["--architecture-only"]
    subprocess.run(args, check=True, env=env)


In [ ]:
import hashlib, shutil
manifest = []
for path in sorted(root.rglob("*")):
    if path.is_file() and path.name != "SHA256SUMS.txt" and "__pycache__" not in path.parts:
        manifest.append(hashlib.sha256(path.read_bytes()).hexdigest()+"  "+str(path.relative_to(root)))
(root/"SHA256SUMS.txt").write_text("\n".join(manifest)+"\n")
output = shutil.make_archive("/content/TAES_R8_Colab_reproduced", "zip", root.parent, root.name)
files.download(output)
